<!-- paper-paired-role-banner -->
> 🟩 **정답 코드 · 오른쪽 창** — 왼쪽과 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 구현 이유와 텐서 shape를 설명해 보세요.

# 15. GCN — Semi-Supervised Classification with Graph Convolutional Networks

- **원 논문:** [Semi-Supervised Classification with Graph Convolutional Networks](https://arxiv.org/abs/1609.02907)
- **저자 / 발표:** Thomas N. Kipf and Max Welling · ICLR (2017)
- **난이도 / 예상 시간:** 중급 · 약 55분
- **선수 지식:** PyTorch 기본, 행렬곱, 그래프 adjacency/degree, cross entropy

이 노트북은 논문 결과 수치를 그대로 재현하는 대규모 benchmark가 아니라,
핵심 수식과 구조가 실제로 어떻게 동작하는지 확인하는 **하드웨어 인식 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하며, 작은 NumPy·Python 실험은
전송 비용을 피하기 위해 CPU에서 실행될 수 있습니다.

## 재현 범위와 완료 기준

**이 노트북이 재현하는 것**

두 community의 작은 합성 그래프에서 self-loop와 대칭 degree normalization을 직접 구현하고, 노드 네 개의 label만으로 2-layer GCN이 나머지 노드를 분류하는지 확인한다.

**원 논문과 다른 것**

논문은 Cora, Citeseer, Pubmed 등의 citation graph에서 sparse 연산을 사용한다. 여기서는 24-node dense adjacency를 사용하지만 Eq. (2), Eq. (8)–(10)의 전파와 masked loss는 같다.

완료하려면 TODO를 구현하고, 검증 셀의 `assert`를 모두 통과시키고, 마지막 관찰 질문에
자신의 말로 답하세요. 수치가 논문과 다르더라도 핵심 불변식과 비교 방향이 맞으면 성공입니다.

## 논문 ↔ 노트북 지도

        절 번호와 수식 번호는 위 링크의 대표 공개본을 기준으로 합니다. 판본에 따라 페이지는 달라질 수
        있으므로 **절 제목과 수식 번호**를 함께 사용하세요.

        | 원 논문의 위치 | 이 노트북의 대응 실습 | 확인할 증거 |
        |---|---|---|
        | §2, Eq. (2) | `GraphConvolution.forward`의 `A_hat @ X @ W` 전파 | node 수를 보존하는 layer 출력 shape와 finite gradient를 검사 |
| §2.2, Eq. (7)–(8) | `normalize_adjacency`의 self-loop와 대칭 degree normalization | 정규화 행렬의 shape·대칭성·유한값을 검사 |
| §3.1, Eq. (9)–(10) | `TinyGCN`의 2-layer propagation과 train mask cross entropy | 미공개 test node 정확도와 학습 loss 감소를 검사 |

        > 읽기 순서: 먼저 해당 절을 훑고 → 코드를 예측해 작성하고 → 출력이 논문의 주장과 왜
        > 연결되는지 한 문장으로 설명합니다.

## 핵심 수식과 구현 설계

$$
H^{(l+1)}=\sigma\!\left(
\hat D^{-1/2}\hat A\hat D^{-1/2}H^{(l)}W^{(l)}\right),
\qquad \hat A=A+I
$$

- $\hat A$는 self-loop를 추가한 adjacency, $\hat D$는 degree matrix입니다.
- symmetric normalization은 이웃 수가 다른 node의 message 크기를 보정합니다.
- adjacency 준비, normalization, graph convolution layer, node loss를 직접 분리합니다.
- $H^{(l)}:[N,D_l]$는 normalized adjacency $[N,N]$와 가중치 $[D_l,D_{l+1}]$를 거칩니다.

## 구현 선택 이유

- **원문에서 보존한 부분:** §2, Eq. (2)
- **노트북 구현:** `GraphConvolution.forward`의 `A_hat @ X @ W` 전파
- **이렇게 구현한 이유:** 완성된 고수준 model을 한 줄로 호출하면
  논문의 핵심 shape·mask·loss·상태 변화가 숨겨집니다. 따라서 이 계산을
  이름이 드러나는 class와 함수로 나누고 중간 tensor를 assertion으로
  검사합니다.
- **실패를 잡는 증거:** node 수를 보존하는 layer 출력 shape와 finite gradient를 검사를 출력 크기만 아니라
  구현 불변식으로 사용합니다.
- **축소 선택:** 논문은 Cora, Citeseer, Pubmed 등의 citation graph에서 sparse 연산을 사용한다. 여기서는 24-node dense adjacency를 사용하지만 Eq. (2), Eq. (8)–(10)의 전파와 masked loss는 같다.는 기본 실행에서 생략합니다.
  이는 Windows CPU에서도 반복 실험이 가능하게 하면서 핵심 메커니즘은
  유지하기 위한 선택입니다.

## 1. Citation graph를 두 community로 축소하기

논문 §2 Eq. (2)의 층은

$$H^{(l+1)}=\sigma(\tilde D^{-1/2}\tilde A\tilde D^{-1/2}H^{(l)}W^{(l)})$$

입니다. §2.2 Eq. (8)의 `renormalization trick`은 $\tilde A=A+I$를 사용합니다.
여기서는 두 community를 가진 24-node graph와 약한 noisy feature를 만들고, 각 class에서
단 두 노드만 label로 공개합니다. 그래프 이웃 집계가 약한 feature 신호를 매끄럽게 만듭니다.

**성공 기준:** 공개하지 않은 20개 노드의 정확도가 85%보다 커야 합니다.

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** A [N,N], X [N,D] → graph hidden [N,H] → node logits [N,K]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(150)
np.random.seed(150)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
# This 24-node dense example may be faster with AI_LAB_DEVICE=cpu;
# auto acceleration preserves the same code path for larger graphs.

n_per_class = 12
n_nodes = 2 * n_per_class
labels = torch.tensor([0] * n_per_class + [1] * n_per_class, dtype=torch.long)
adjacency = torch.zeros(n_nodes, n_nodes)

# 각 community 안에서 ring의 1-hop/2-hop 이웃을 연결한다.
for offset in (0, n_per_class):
    for i in range(n_per_class):
        for delta in (1, 2):
            j = offset + (i + delta) % n_per_class
            k = offset + i
            adjacency[k, j] = adjacency[j, k] = 1.0
# 소수의 community 간 edge는 실제 그래프의 noise 역할을 한다.
for left, right in ((2, 14), (8, 20)):
    adjacency[left, right] = adjacency[right, left] = 1.0

feature_rng = torch.Generator().manual_seed(151)
weak_signal = (2 * labels.float() - 1.0)[:, None] * 0.45
features = torch.cat(
    [
        weak_signal + 0.65 * torch.randn(n_nodes, 1, generator=feature_rng),
        0.75 * torch.randn(n_nodes, 2, generator=feature_rng),
        torch.ones(n_nodes, 1),
    ],
    dim=1,
)
train_idx = torch.tensor([0, 5, 12, 17])
test_mask = torch.ones(n_nodes, dtype=torch.bool)
test_mask[train_idx] = False
labels, adjacency, features, train_idx, test_mask = ACCELERATOR.move(
    labels, adjacency, features, train_idx, test_mask
)

assert torch.allclose(adjacency, adjacency.T)
assert features.shape == (24, 4)
print(ACCELERATOR.summary())
edge_count = int(adjacency.sum().item() / 2)
print(f"nodes={n_nodes}, undirected edges={edge_count}, labels used=4")

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §2.2, Eq. (7)–(8)
- **이 셀의 책임:** `normalize_adjacency`의 self-loop와 대칭 degree normalization
- **Tensor shape 계약:** A [N,N], X [N,D] → graph hidden [N,H] → node logits [N,K]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 정규화 행렬의 shape·대칭성·유한값을 검사. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def normalize_adjacency(adjacency):
    # §2.2 Eq. (8): self-loop를 먼저 넣은 뒤 degree로 대칭 정규화한다.
    a_tilde = adjacency + torch.eye(
        adjacency.shape[0], dtype=adjacency.dtype, device=adjacency.device
    )
    degree = a_tilde.sum(dim=1)
    inv_sqrt_degree = degree.clamp_min(1e-12).pow(-0.5)
    return inv_sqrt_degree[:, None] * a_tilde * inv_sqrt_degree[None, :]


a_hat = normalize_adjacency(adjacency)
assert a_hat.shape == adjacency.shape
assert torch.allclose(a_hat, a_hat.T, atol=1e-6)
assert torch.isfinite(a_hat).all()
assert torch.all(a_hat.diag() > 0), "self-loop가 모든 node에 있어야 한다"
print("renormalized adjacency 검증 통과")

### 구현 단계 3 · 핵심 연산·모델

- **원문 위치:** §2, Eq. (2) / §3.1, Eq. (9)–(10)
- **이 셀의 책임:** `GraphConvolution.forward`의 `A_hat @ X @ W` 전파 / `TinyGCN`의 2-layer propagation과 train mask cross entropy
- **Tensor shape 계약:** A [N,N], X [N,D] → graph hidden [N,H] → node logits [N,K]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** node 수를 보존하는 layer 출력 shape와 finite gradient를 검사 / 미공개 test node 정확도와 학습 loss 감소를 검사. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
class GraphConvolution(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(in_features, out_features))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, x, normalized_adjacency):
        return normalized_adjacency @ x @ self.weight


class TinyGCN(nn.Module):
    def __init__(self):
        super().__init__()
        self.gcn1 = GraphConvolution(4, 12)
        self.gcn2 = GraphConvolution(12, 2)

    def forward(self, x, normalized_adjacency):
        hidden = F.relu(self.gcn1(x, normalized_adjacency))
        return self.gcn2(hidden, normalized_adjacency)


gcn = TinyGCN().to(DEVICE)
logits = gcn(features, a_hat)
assert logits.shape == (n_nodes, 2)
assert sum(parameter.numel() for parameter in gcn.parameters()) == 72

### 구현 단계 4 · 목적함수·학습 update

- **원문 위치:** §3.1, Eq. (9)–(10)
- **이 셀의 책임:** `TinyGCN`의 2-layer propagation과 train mask cross entropy
- **Tensor shape 계약:** A [N,N], X [N,D] → graph hidden [N,H] → node logits [N,K]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 미공개 test node 정확도와 학습 loss 감소를 검사. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
optimizer = torch.optim.Adam(gcn.parameters(), lr=0.03, weight_decay=5e-4)
loss_history = []
with torch.no_grad():
    initial_logits = gcn(features, a_hat)[train_idx]
    initial_loss = float(F.cross_entropy(initial_logits, labels[train_idx]))

for epoch in range(250):
    logits = gcn(features, a_hat)
    # §3.1 Eq. (10): Y_L, 즉 label이 공개된 node만 loss에 들어간다.
    loss = F.cross_entropy(logits[train_idx], labels[train_idx])
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    loss_history.append(float(loss.detach()))

print(f"masked CE: {initial_loss:.4f} -> {loss_history[-1]:.4f}")
assert len(loss_history) == 250
assert math.isfinite(loss_history[-1]) and loss_history[-1] < initial_loss

### 구현 단계 5 · 평가·완료 증거

- **원문 위치:** §3.1, Eq. (9)–(10)
- **이 셀의 책임:** `TinyGCN`의 2-layer propagation과 train mask cross entropy
- **Tensor shape 계약:** A [N,N], X [N,D] → graph hidden [N,H] → node logits [N,K]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** 미공개 test node 정확도와 학습 loss 감소를 검사. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
gcn.eval()
with torch.no_grad():
    final_logits = gcn(features, a_hat)
    predictions = final_logits.argmax(dim=1)
train_accuracy = float((predictions[train_idx] == labels[train_idx]).float().mean())
test_accuracy = float((predictions[test_mask] == labels[test_mask]).float().mean())
print(
    f"labeled accuracy={train_accuracy:.3f}, held-out node accuracy={test_accuracy:.3f}"
)
assert train_accuracy == 1.0
assert test_accuracy > 0.85

angles = np.linspace(0, 2 * np.pi, n_per_class, endpoint=False)
positions = np.vstack(
    [
        np.c_[np.cos(angles) - 1.35, np.sin(angles)],
        np.c_[np.cos(angles) + 1.35, np.sin(angles)],
    ]
)
train_idx_cpu = train_idx.detach().cpu().numpy()


def draw_graph(axis, colors, title):
    edge_i, edge_j = torch.where(torch.triu(adjacency, diagonal=1) > 0)
    for i, j in zip(edge_i.tolist(), edge_j.tolist()):
        axis.plot(
            positions[[i, j], 0],
            positions[[i, j], 1],
            color="0.82",
            lw=0.8,
            zorder=0,
        )
    axis.scatter(
        positions[:, 0],
        positions[:, 1],
        c=colors,
        cmap="coolwarm",
        s=55,
        zorder=2,
    )
    axis.scatter(
        positions[train_idx_cpu, 0],
        positions[train_idx_cpu, 1],
        facecolors="none",
        edgecolors="black",
        marker="s",
        s=120,
        linewidths=1.4,
        label="labeled",
    )
    axis.set_title(title)
    axis.set_aspect("equal")
    axis.axis("off")
    axis.legend(loc="upper center")


fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
draw_graph(axes[0], labels.detach().cpu().numpy(), "ground truth (4 labels exposed)")
draw_graph(axes[1], predictions.detach().cpu().numpy(), "GCN predictions")
axes[2].plot(loss_history)
axes[2].set(title="masked cross entropy", xlabel="epoch", ylabel="loss")
fig.tight_layout()
plt.show()

## 7. 해석 예시 답안

1. 해당 층의 출력은 이웃 feature만 집계하고 자기 feature의 직접 경로를 잃습니다. $A+I$는 이를 보존합니다.
2. 평가할 노드의 정답까지 gradient에 사용하므로 label이 일부만 주어진 전이적
   node classification 조건과 test 분리가 깨집니다.
3. 여기서는 $24	imes24$ dense 행렬곱이라 일반적으로 $O(N^2)$ 저장/계산을
   합니다. 논문은 sparse adjacency를 사용해 한 층의 전파를 edge 수에
   선형인 $O(|E|)$로 수행합니다.

## 마무리 해설과 다음 실험

1. 핵심 연산과 구현 위치를 원 논문의 용어로 다시 연결해 설명하세요.
2. 이 미니 실험은 `논문은 Cora, Citeseer, Pubmed 등의 citation graph에서 sparse 연산을 사용한다. 여기서는 24-node dense adjacency를 사용하지만 Eq. (2), Eq. (8)–(10)의 전파와 masked loss는 같다.`를 재현하지 않으므로 논문의 최종 성능을 검증한 것이 아닙니다.
3. 다음에는 데이터 크기, 모델 폭, 학습 횟수 중 **하나만** 바꾸고 같은 seed에서 metric을 비교하세요.

정답의 수치 자체보다, shape·확률 정규화·mask·gradient 같은 불변식을 먼저 확인하는 습관이
논문 구현에서 가장 강한 디버깅 도구입니다.